# Oracle seed-11 restore (ONLY this experiment)

Restores `L_upQ_component0_seed11` **`test_results.p`** — lost in a drive merge — so the
oracle **log-NSE / KGE** columns at seed 11 can be filled (`PAPER_TABLE.md`,
`METRIC_HONESTY.md`). Seeds 13/17 are already complete; this is the one missing piece.

**Why a dedicated notebook:** the generic runner skips when `test_metrics.csv` exists — and
for this run the metrics file survived the merge while `test_results.p` did NOT. So the
correct idempotency key here is **`test_results.p`**: train only if there is no checkpoint,
then ALWAYS re-evaluate (cheap) to regenerate `test_results.p`.

**Path of least friction: Runtime → Change runtime type → T4 GPU → Run all.** Idempotent —
re-running after it is done is a no-op.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (find CAMELS on Drive; set runs dir)

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # leave blank to auto-detect
SEED=11
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('restoring L_upQ seed', SEED)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4 like the other notebooks)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build the observed upstream-Q feature (full graph)

The `L_upQ` config points at `features/upstream_q_component0_lag1.p`, which is gitignored
(so not in the clone). Regenerate it. Idempotent: skips if already present with a named index.

In [ ]:
%cd {REPO_DIR}
import pickle
feat_p='experiments/topology_ablation/features/upstream_q_component0_lag1.p'
ok=False
if os.path.isfile(feat_p):
    d=pickle.load(open(feat_p,'rb')); ok=(d[next(iter(d))].index.name=='date')
if ok:
    print('[skip] observed upstream_q feature already built')
else:
    !python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -1
    !python experiments/topology_ablation/build_upstream_discharge_feature.py --network component0 --lag-days 1 2>&1 | tail -1

## Cell 8 — Train (only if no checkpoint) then ALWAYS evaluate → regenerate test_results.p

Idempotency key here is **`test_results.p`** (NOT `test_metrics.csv` — that file survived the
merge and would falsely trigger a skip). Train only if the epoch-30 checkpoint is missing;
evaluate is cheap and always run to (re)produce `test_results.p`.

In [ ]:
%cd {REPO_DIR}
import glob, yaml
from pathlib import Path
name=f'L_upQ_component0_seed{SEED}'
run_dir=f'{REPO_DIR}/runs/topology_ablation/component0/{name}'
ckpt=f'{run_dir}/model_epoch030.pt'
results_p=f'{run_dir}/test/model_epoch030/test_results.p'
FEAT=os.path.abspath('experiments/topology_ablation/features/upstream_q_component0_lag1.p')

if os.path.isfile(results_p):
    print(f'[skip] {name} test_results.p already present — nothing to do')
else:
    if not os.path.isfile(ckpt):
        # write the config (mirrors the committed L_upQ_component0_seed11.yaml, cuda) and TRAIN
        base=yaml.safe_load(open('experiments/topology_ablation/configs/L_upQ_component0_seed11.yaml'))
        base.update(experiment_name=name, run_dir='runs/topology_ablation/component0',
                    device='cuda:0', seed=SEED,
                    additional_feature_files=[FEAT])
        cfg_out='experiments/topology_ablation/configs/_restore_L_upQ_seed{}.yaml'.format(SEED)
        yaml.safe_dump(base, open(cfg_out,'w'), sort_keys=False)
        !python neuralhydrology/nh_run.py train --config-file {cfg_out} 2>&1 | tail -3
        # rename timestamped dir -> canonical
        cands=sorted(glob.glob(f'{run_dir}_*'))
        if cands: os.rename(cands[-1], run_dir)
    else:
        print(f'[keep] checkpoint exists; skipping train, evaluating only')
    # ALWAYS evaluate -> regenerates test_results.p (+ test_metrics.csv)
    !python neuralhydrology/nh_run.py evaluate --run-dir {run_dir} --epoch 30 2>&1 | tail -2

## Cell 9 — Verdict: determinism check + oracle metrics now available

In [ ]:
%cd {REPO_DIR}
import pandas as pd, pickle, numpy as np
name=f'L_upQ_component0_seed{SEED}'
base=f'{REPO_DIR}/runs/topology_ablation/component0'
md=f'{base}/{name}/test/model_epoch030/test_metrics.csv'
rp=f'{base}/{name}/test/model_epoch030/test_results.p'

# Determinism check. Tolerance is +-0.01 (the cross-seed std), NOT +-0.005 — NH is seed-fixed
# but not bitwise-deterministic across different Colab GPU/library images. The recorded 0.703 is
# itself a drive-merge "recorded original". Reproduction WITHIN NOISE is the pass condition.
m=pd.read_csv(md); med=m['NSE'].median()
tol=0.01
print(f'oracle seed{SEED} median NSE = {med:.4f}')
print(f'  vs recorded 0.703 -> |Δ|={abs(med-0.703):.4f}; reproduces within cross-env tol ±{tol}: {abs(med-0.703)<=tol}')
print(f'  oracle Δ vs L(0.653) = {med-0.653:+.4f}  (headline oracle is +0.035..0.037 -> consistent)')
print(f'test_results.p present: {os.path.isfile(rp)}  (the file that was missing — now restored)')
print('\nNOTE: do NOT read log-NSE off a quick inline calc here — use the vetted')
print('analyze_metric_honesty.py over this restored results.p (per-basin NaN-mask + eps handling).')

## Cell 10 — Vetted log-NSE + PERSISTENCE CHECK (did it reach Drive?)

Runs the committed `analyze_metric_honesty.py` (the trustworthy log-NSE pipeline) now that the
oracle results.p exists, then **verifies the run physically landed in Drive** and force-flushes.
If the persistence check fails, the run is only in the VM and will be lost on recycle.

In [ ]:
%cd {REPO_DIR}
import os, subprocess
name=f'L_upQ_component0_seed{SEED}'
run_dir=f'{REPO_DIR}/runs/topology_ablation/component0/{name}'
rp=f'{run_dir}/test/model_epoch030/test_results.p'
# 1) vetted log-NSE: analyze_metric_honesty reads L/L_upQ/L_upQpred results.p across seeds
!python experiments/topology_ablation/analyze_metric_honesty.py 2>&1 | sed -n '1,20p'
# 2) PERSISTENCE CHECK — is the run physically in Drive (not just the VM)?
drive_path=f'{DRIVE_RUNS}/topology_ablation/component0/{name}/test/model_epoch030/test_results.p'
print('\n=== persistence check ===')
print('results.p in VM   :', os.path.isfile(rp))
print('results.p in Drive:', os.path.isfile(drive_path), '\n ', drive_path)
if os.path.isfile(rp) and not os.path.isfile(drive_path):
    # runs/ is a symlink to Drive; if the symlinked target isn't showing the file, force a sync
    print('  -> forcing Drive flush...')
    from google.colab import drive as _d; _d.flush_and_unmount(); _d.mount('/content/drive')
    print('  re-check after flush:', os.path.isfile(drive_path))
print('\nIf Drive shows True, the run is safe. If False, the symlink did not point at Drive —')
print('download the folder manually:  from google.colab import files; files.download(rp)')


## Done

Run persists to Drive (`neural_hydrology_runs/topology_ablation/component0/L_upQ_component0_seed11`).
Only this one experiment ran. To fill the paper table locally: copy that folder's
`test/model_epoch030/` back into the repo, then `python experiments/topology_ablation/build_paper_table.py`
and `python experiments/topology_ablation/analyze_metric_honesty.py`.